# Cross-Factory TTA Experiment

Complete experiment: train YOLOv8m on SH17, evaluate on Pictor-PPE, apply TENT adaptation.

Supports:
- Kaggle (dual T4) and Colab (single GPU) presets
- Auto-resume from checkpoint if interrupted
- All code inline

## Setup

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics", "pyyaml", "pillow", "tqdm"
])

print("Dependencies installed.")

In [ ]:
from pathlib import Path
import os
import shutil
import json
import copy

import torch
import torch.nn as nn
from ultralytics import YOLO
from PIL import Image

print("Imports OK.")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}")

## Config

**EDIT THIS SECTION** for your environment:

- **Kaggle**: Set `PLATFORM = "kaggle"`, update dataset paths to `/kaggle/input/datasets/...`
- **Colab**: Set `PLATFORM = "colab"`, update dataset paths to your mounted drive

In [ ]:
# === EDIT THIS SECTION ===

# Platform: "kaggle" or "colab"
PLATFORM = "kaggle"

# Dataset paths (update to your mounts)
if PLATFORM == "kaggle":
    SH17_DIR = Path("/kaggle/input/datasets/mugheesahmad/sh17-dataset-for-ppe-detection")
    PICTOR_DIR = Path("/kaggle/input/datasets/zyanahmed/pictor-ppe")
    WORK_DIR = Path("/kaggle/working")
else:  # colab
    SH17_DIR = Path("/content/drive/MyDrive/datasets/sh17")  # Update this
    PICTOR_DIR = Path("/content/drive/MyDrive/datasets/pictor_ppe")  # Update this
    WORK_DIR = Path("/content")

DATA_DIR = WORK_DIR / "data"
RUNS_DIR = WORK_DIR / "runs"

# === END EDIT SECTION ===

print(f"Platform: {PLATFORM}")
print(f"SH17_DIR exists: {SH17_DIR.exists()}")
print(f"PICTOR_DIR exists: {PICTOR_DIR.exists()}")
print(f"Work dir: {WORK_DIR}")

In [ ]:
# Training hyperparameters (auto-adjusted per platform)

if PLATFORM == "kaggle":
    # Dual T4 preset
    TRAIN_CONFIG = {
        "device": "0,1",
        "batch": 32,
        "workers": 8,
        "lr0": 0.02,  # scaled for larger batch
        "warmup_epochs": 5,
    }
else:
    # Colab single GPU preset (T4/V100/A100)
    TRAIN_CONFIG = {
        "device": 0,
        "batch": 16,
        "workers": 4,
        "lr0": 0.01,
        "warmup_epochs": 3,
    }

# Common hyperparameters
TRAIN_CONFIG.update({
    "epochs": 50,
    "imgsz": 640,
    "optimizer": "SGD",
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "amp": True,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "translate": 0.1,
    "scale": 0.5,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "verbose": False,
    "exist_ok": True,
})

print(f"Training config: batch={TRAIN_CONFIG['batch']}, device={TRAIN_CONFIG['device']}, lr0={TRAIN_CONFIG['lr0']}")

## Data Preparation

In [ ]:
SH17_CLASSES = [
    "Coverall", "Face Shield", "Gloves", "Goggles", "Hard Hat",
    "No Coverall", "No Face Shield", "No Gloves", "No Goggles", "No Hard Hat",
    "No Safety Boot", "No Safety Vest", "No Vest", "Safety Boot", "Safety Vest",
    "Vest", "person",
]

PICTOR_CLASSES = ["helmet", "head", "person"]


def setup_sh17():
    print("\n=== Setting up SH17 ===")
    src_images = SH17_DIR / "images"
    src_labels = SH17_DIR / "labels"
    
    for split in ("train", "val"):
        out_images = DATA_DIR / "sh17" / "images" / split
        out_labels = DATA_DIR / "sh17" / "labels" / split
        out_images.mkdir(parents=True, exist_ok=True)
        out_labels.mkdir(parents=True, exist_ok=True)
        
        split_file = SH17_DIR / f"{split}_files.txt"
        filenames = split_file.read_text().splitlines()
        filenames = [f.strip() for f in filenames if f.strip()]
        print(f"  {split}: {len(filenames)} files")
        
        for fname in filenames:
            img_src = src_images / fname
            stem = Path(fname).stem
            label_src = src_labels / f"{stem}.txt"
            
            if img_src.exists():
                shutil.copy2(img_src, out_images / fname)
            if label_src.exists():
                shutil.copy2(label_src, out_labels / f"{stem}.txt")
    
    print(f"  SH17 train: {len(list((DATA_DIR / 'sh17/images/train').glob('*')))} images")
    print(f"  SH17 val: {len(list((DATA_DIR / 'sh17/images/val').glob('*')))} images")


def setup_pictor():
    print("\n=== Setting up Pictor-PPE ===")
    src_images = PICTOR_DIR / "images" / "test"
    src_labels = PICTOR_DIR / "labels" / "test"
    
    out_images = DATA_DIR / "pictor_ppe" / "images" / "test"
    out_labels = DATA_DIR / "pictor_ppe" / "labels" / "test"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)
    
    if src_images.exists() and src_labels.exists():
        shutil.copytree(src_images, out_images, dirs_exist_ok=True)
        shutil.copytree(src_labels, out_labels, dirs_exist_ok=True)
        print(f"  Pictor test: {len(list(out_images.glob('*')))} images")
    else:
        print(f"  [WARN] Pictor source not found at {src_images}")


def write_yamls():
    print("\n=== Writing YAML configs ===")
    
    sh17_yaml = DATA_DIR / "sh17" / "sh17.yaml"
    sh17_yaml.write_text(f"""path: {DATA_DIR / 'sh17'}
train: images/train
val: images/val

nc: {len(SH17_CLASSES)}
names: {SH17_CLASSES}
""")
    
    pictor_yaml = DATA_DIR / "pictor_ppe" / "pictor.yaml"
    pictor_yaml.write_text(f"""path: {DATA_DIR / 'pictor_ppe'}
train: images/test
val: images/test

nc: {len(PICTOR_CLASSES)}
names: {PICTOR_CLASSES}
""")
    
    print(f"  SH17 yaml: {sh17_yaml}")
    print(f"  Pictor yaml: {pictor_yaml}")


if (DATA_DIR / "sh17" / "sh17.yaml").exists():
    print("Data already prepared.")
else:
    setup_sh17()
    setup_pictor()
    write_yamls()
    print("\nData preparation complete.")

## Training on SH17

Auto-resumes from checkpoint if available.

In [ ]:
# Check for existing checkpoint
RUN_NAME = "sh17_yolov8m"
CHECKPOINT_PATH = RUNS_DIR / "train" / RUN_NAME / "weights" / "last.pt"
BEST_WEIGHTS_PATH = RUNS_DIR / "train" / RUN_NAME / "weights" / "best.pt"

if CHECKPOINT_PATH.exists():
    print(f"Found checkpoint: {CHECKPOINT_PATH}")
    print("Resuming training...")
    model = YOLO(str(CHECKPOINT_PATH))
    results = model.train(resume=True)
else:
    print("Starting fresh training...")
    
    # Compact epoch logger
    def compact_epoch_callback(trainer):
        try:
            loss_items = getattr(trainer, "loss_items", None)
            box = cls = dfl = None
            if loss_items is not None and hasattr(loss_items, "__len__") and len(loss_items) >= 3:
                box, cls, dfl = float(loss_items[0]), float(loss_items[1]), float(loss_items[2])
            
            metrics = getattr(trainer, "metrics", {})
            if not isinstance(metrics, dict):
                metrics = getattr(metrics, "results_dict", {})
            
            map50 = metrics.get("metrics/mAP50(B)") or metrics.get("metrics/mAP50")
            map5095 = metrics.get("metrics/mAP50-95(B)") or metrics.get("metrics/mAP50-95")
            
            epoch = int(getattr(trainer, "epoch", -1)) + 1
            epochs = int(getattr(trainer, "epochs", -1))
            
            parts = [f"epoch {epoch}/{epochs}"]
            if box is not None: parts.append(f"box={box:.3f}")
            if cls is not None: parts.append(f"cls={cls:.3f}")
            if dfl is not None: parts.append(f"dfl={dfl:.3f}")
            if map50 is not None: parts.append(f"mAP50={float(map50):.3f}")
            if map5095 is not None: parts.append(f"mAP50-95={float(map5095):.3f}")
            
            print(" | ".join(parts))
        except Exception as e:
            print(f"[WARN] Callback error: {e}")
    
    print("epoch/epochs | box | cls | dfl | mAP50 | mAP50-95")
    
    model = YOLO("yolov8m.pt")
    model.add_callback("on_fit_epoch_end", compact_epoch_callback)
    
    results = model.train(
        data=str(DATA_DIR / "sh17" / "sh17.yaml"),
        name=RUN_NAME,
        project=str(RUNS_DIR / "train"),
        **TRAIN_CONFIG
    )

print(f"\nTraining complete.")
print(f"Best weights: {BEST_WEIGHTS_PATH}")
if BEST_WEIGHTS_PATH.exists():
    print(f"mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
    print(f"mAP50-95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

## Baseline Evaluation (Zero-Shot)

In [ ]:
print("\n=== Baseline Zero-Shot Evaluation ===")

baseline_model = YOLO(str(BEST_WEIGHTS_PATH))
baseline_metrics = baseline_model.val(
    data=str(DATA_DIR / "pictor_ppe" / "pictor.yaml"),
    imgsz=640,
    batch=16,
    split="val",
    conf=0.001,
    iou=0.6,
    name="baseline_pictor",
    project=str(RUNS_DIR / "eval"),
    exist_ok=True,
    verbose=False,
)

baseline_map50 = float(baseline_metrics.box.map50)
baseline_map5095 = float(baseline_metrics.box.map)

print(f"Baseline mAP50: {baseline_map50:.4f}")
print(f"Baseline mAP50-95: {baseline_map5095:.4f}")

baseline_results = {
    "mAP50": baseline_map50,
    "mAP50_95": baseline_map5095,
    "per_class_AP50": {
        name: float(ap)
        for name, ap in zip(baseline_metrics.names.values(), baseline_metrics.box.ap50)
    },
}

del baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## TENT Adaptation

In [ ]:
# TENT implementation

def configure_model_for_tent(model: nn.Module) -> nn.Module:
    model.train()
    for param in model.parameters():
        param.requires_grad_(False)
    
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            m.requires_grad_(True)
            m.track_running_stats = False
            m.running_mean = None
            m.running_var = None
    
    return model


def collect_bn_params(model: nn.Module):
    params, names = [], []
    for nm, m in model.named_modules():
        if isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            for pnm, p in m.named_parameters():
                if p.requires_grad:
                    params.append(p)
                    names.append(f"{nm}.{pnm}")
    return params, names


def softmax_entropy(logits: torch.Tensor) -> torch.Tensor:
    probs = logits.softmax(dim=-1)
    return -(probs * (probs + 1e-8).log()).sum(dim=-1)


def tent_loss(predictions) -> torch.Tensor:
    if isinstance(predictions, (list, tuple)):
        entropy_terms = []
        for pred in predictions:
            if not isinstance(pred, torch.Tensor):
                continue
            if pred.ndim == 3 and pred.shape[1] > 4:
                cls_logits = pred[:, 4:, :]
                cls_logits = cls_logits.permute(0, 2, 1)
                entropy_terms.append(softmax_entropy(cls_logits))
        if entropy_terms:
            return torch.cat([e.reshape(-1) for e in entropy_terms]).mean()
    return torch.tensor(0.0, requires_grad=True)


class TENT:
    def __init__(self, model: YOLO, lr: float = 0.001, steps: int = 1):
        self.model = model
        self.steps = steps
        self.lr = lr
        
        self._original_state = copy.deepcopy(model.model.state_dict())
        configure_model_for_tent(model.model)
        params, param_names = collect_bn_params(model.model)
        print(f"[TENT] Adapting {len(params)} BN parameters")
        
        self.optimizer = torch.optim.SGD(params, lr=lr, momentum=0.9)
    
    @torch.enable_grad()
    def step(self, batch_paths: list[str]) -> list:
        for _ in range(self.steps):
            results_raw = self.model.model(self._preprocess(batch_paths))
            loss = tent_loss(results_raw)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
        
        return self.model.predict(batch_paths, verbose=False)
    
    def _preprocess(self, paths: list[str]) -> torch.Tensor:
        from ultralytics.data.augment import LetterBox
        import numpy as np
        import cv2
        
        tensors = []
        lb = LetterBox(new_shape=(640, 640))
        for p in paths:
            img = cv2.imread(str(p))
            img = lb(image=img)
            img = img[:, :, ::-1].transpose(2, 0, 1).copy()
            tensors.append(torch.from_numpy(img).float() / 255.0)
        batch = torch.stack(tensors)
        device = next(self.model.model.parameters()).device
        return batch.to(device)


print("TENT implementation ready.")

In [ ]:
print("\n=== TENT Adaptation on Pictor ===")

tent_model = YOLO(str(BEST_WEIGHTS_PATH))
adapter = TENT(tent_model, lr=0.001, steps=1)

def on_val_batch_start(validator):
    batch_imgs = validator.batch.get("im_file", [])
    if batch_imgs:
        adapter.step(list(batch_imgs))

tent_model.add_callback("on_val_batch_start", on_val_batch_start)

tent_metrics = tent_model.val(
    data=str(DATA_DIR / "pictor_ppe" / "pictor.yaml"),
    imgsz=640,
    batch=16,
    split="val",
    conf=0.001,
    iou=0.6,
    name="tent_pictor",
    project=str(RUNS_DIR / "eval"),
    exist_ok=True,
    verbose=False,
)

tent_map50 = float(tent_metrics.box.map50)
tent_map5095 = float(tent_metrics.box.map)
recovery = tent_map50 - baseline_map50

print(f"TENT mAP50: {tent_map50:.4f}")
print(f"TENT mAP50-95: {tent_map5095:.4f}")
print(f"Recovery: {recovery:+.4f}")

## Results Summary

In [ ]:
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Baseline (source-only) mAP50:  {baseline_map50:.4f}")
print(f"TENT (adapted) mAP50:          {tent_map50:.4f}")
print(f"Recovery:                      {recovery:+.4f}")
print("="*60)

final_results = {
    "platform": PLATFORM,
    "training_config": TRAIN_CONFIG,
    "baseline_mAP50": baseline_map50,
    "baseline_mAP50_95": baseline_map5095,
    "tent_mAP50": tent_map50,
    "tent_mAP50_95": tent_map5095,
    "recovery_mAP50": recovery,
    "baseline_per_class": baseline_results["per_class_AP50"],
    "tent_per_class": {
        name: float(ap)
        for name, ap in zip(tent_metrics.names.values(), tent_metrics.box.ap50)
    },
}

results_path = WORK_DIR / f"results_{PLATFORM}.json"
results_path.write_text(json.dumps(final_results, indent=2))
print(f"\nSaved to: {results_path}")